# Feature Attribution using Ranking - v1.4

In [41]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost
!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow

     |████████████████████████████████| 511.7 MB 12 kB/s eta 0:00:011    |██                              | 31.3 MB 297 kB/s eta 0:26:55     |██████████▌                     | 168.1 MB 215 kB/s eta 0:26:33     |████████████                    | 191.7 MB 194 kB/s eta 0:27:29     |█████████████████▋              | 281.7 MB 269 kB/s eta 0:14:14     |██████████████████              | 289.1 MB 122 kB/s eta 0:30:19     |███████████████████▉            | 317.7 MB 276 kB/s eta 0:11:42     |███████████████████████         | 368.2 MB 208 kB/s eta 0:11:29     |████████████████████████████    | 446.4 MB 341 kB/s eta 0:03:12     |████████████████████████████▏   | 450.8 MB 162 kB/s eta 0:06:15     |██████████████████████████████  | 480.2 MB 119 kB/s eta 0:04:23     |██████████████████████████████▊ | 490.5 MB 242 kB/s eta 0:01:28
     |████████████████████████████████| 65 kB 94 kB/s  eta 0:00:01
     |████████████████████████████████| 5.8 MB 145 kB/s eta 0:00:01
     |████████████████████████████████

     |████████████████████████████████| 77 kB 127 kB/s eta 0:00:01
     |████████████████████████████████| 151 kB 184 kB/s eta 0:00:01
  Created wheel for termcolor: filename=termcolor-1.1.0-py3-none-any.whl size=4829 sha256=ebfc7befa1944e78941c756e23862997d2fe4248574c19bfb561505804d27f00
  Stored in directory: /home/evortigosa/.cache/pip/wheels/a0/16/9c/5473df82468f958445479c59e784896fa24f4a5fc024b0f501
Successfully built termcolor
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib-metadata 3.10.0
    Uninstalling importlib-metadata-3.10.0:
      Successfully uninstalled importlib-metadata-3.10.0


In [2]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
from catboost import CatBoostClassifier

from scipy.sparse.linalg import eigs

import networkx as nx
import GraphPR as gpr
import PageRank as pr

# Data preprocessing methods

In [3]:
# return one df without nan's on num_cols
# numerical features: fill nan's with median/mean/mode

def pre_proc_fillna_num_fts(df,num_cols,num_type='mean'):
    df_train= df.copy()

    if(num_type=='median'):
        for col in num_cols:
            ft_median= df_train[col].median()
            df_train[col]= df_train[col].fillna(ft_median)
    elif(num_type=='mode'):
        for col in num_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)
    else:
        for col in num_cols:
            ft_mean= df_train[col].mean()
            df_train[col]= df_train[col].fillna(ft_mean)

    return df_train

In [4]:
# return one df without nan's on cat_cols
# categorical features: fill nan's with mode/mean/median

def pre_proc_fillna_cat_fts(df,cat_cols,cat_type='mode'):
    df_train= df.copy()
    
    if(cat_type!='mode' and type(df_train[cat_cols[0]].value_counts().index[0])!=type('str')):
        if(cat_type=='mean'):
            for col in cat_cols:
                ft_mean= df_train[col].mean()
                df_train[col]= df_train[col].fillna(ft_mean)
        elif(cat_type=='median'):
            for col in cat_cols:
                ft_median= df_train[col].median()
                df_train[col]= df_train[col].fillna(ft_median)
    else:
        for col in cat_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)

    return df_train

In [377]:
# selected_cols refers only to df's columns with numerical values
# if selected_cols is not defined as argument, all df columns will be normalized

def normalize_selected(df, selected_cols=[]):
    result= df.copy()
    
    if (len(selected_cols)== 0):
        selected_cols= df.columns
    
    for col in selected_cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

In [6]:
# return a sorted DataFrame with Features and Importances - OHE compacted, that is, dataset's original features

def ft_importance_df(importances, ft_names, replace_list):
    fti= pd.Series(importances, index=ft_names).sort_values(ascending=False).to_frame().reset_index()
    fti= fti.rename(columns= {'index':'Feature',0:'Importance'}, inplace=False)
    fti['Feature'].replace(replace_list, inplace=True)
    fti= fti.groupby(['Feature']).sum().sort_values('Importance', ascending=False).reset_index()
    
    return fti

# Feature Attribution using Ranking - methods

In [7]:
# return a DataFrame with replace values (mean/median/mode/none to categorical and numeric) to fill train cols. df is post-processed (cat_cols encoded)

def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [8]:
# train the ML model and return its mean accuracy after n_train runs

def train_model_get_acc_mean(model, x_trn, x_tst, y_trn, y_tst, n_train):
    trainings= []
    
    np_ar= np.ndarray((0,0))    # model.fit() expect a 1d array instead of a pd.DataFrame
    pd_df= pd.DataFrame(np_ar)  # we need to verify and change it using values.ravel()
    
    if (type(y_trn)== type(pd_df)):
        
        for i in range(n_train):

            model.fit(x_trn, y_trn.values.ravel())
            acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

            trainings.append(acc)
    else:
        for i in range(n_train):

            model.fit(x_trn, y_trn)
            acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

            trainings.append(acc)
            
    return np.mean(trainings)

In [9]:
# re-training is needed because machine learning models typically assume that the train and the test data comes from a similar distribution 
# (Hooker et al., 2018)

# model is a NO trained model

# here we return p(x|i) and p(x|ij)
def remove_and_retrain(model, replace_ft_vals, x_trn, x_tst, y_trn, y_tst, n_train, verbose=False):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_trn.columns)

    start= time.time()

    for i in range(n_fts):
        acc_row= []

        # replace the i-th ft with its respective mode/mean to "remove" it. train the ML model and get the mean accuracy
        train_copy_no_i= x_trn.copy()
        train_copy_no_i.loc[:,train_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(train_model_get_acc_mean(model, train_copy_no_i, x_tst, y_trn, y_tst, n_train))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft with its respective mode/mean to "remove" it. here, we "remove" the i-th and the j-th ft
                # train the ML model and get the mean accuracy
                train_copy_no_ij= train_copy_no_i.copy()
                train_copy_no_ij.loc[:,train_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(train_model_get_acc_mean(model, train_copy_no_ij, x_tst, y_trn, y_tst, n_train))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [12]:
# remove WITH NO retraining - to speedup and stability improvements
# model is a TRAINED model

# here we return p(x|i) and p(x|ij)
def remove_and_return_costs(model, replace_ft_vals, x_tst, y_tst, verbose=False):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_tst.columns)

    start= time.time()
    
    
    np_ar= np.ndarray((0,0))    # model.fit() expect a 1d array instead of a pd.DataFrame
    pd_df= pd.DataFrame(np_ar)  # we need to verify and change it using values.ravel()
    

    for i in range(n_fts):  # remove and predict process to get gains/losses
        acc_row= []

        # replace the i-th ft of test set with its respective mode/mean to "remove" it. 
        x_tst_copy_no_i= x_tst.copy()
        x_tst_copy_no_i.loc[:,x_tst_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst_copy_no_i)))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft of test with its respective mode/mean to "remove" it. 
                # here, we "remove" the i-th and the j-th ft
                # test the ML model and get the mean accuracy
                x_tst_copy_no_ij= x_tst_copy_no_i.copy()
                x_tst_copy_no_ij.loc[:,x_tst_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst_copy_no_ij)))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [10]:
import math
from sklearn.neighbors import NearestNeighbors

# find the test_size nearest neighbors from a target_x instance
# return train, test, labels_train, labels_test datsets based on knn
# target_x and target_y are into testX and testY

def knn_train_test_split(df_x, df_y, target_x, target_y, test_size= 0.2):
    
    m_ins= df_x.shape[0] # m x n dataset

    neighbors= math.floor(test_size* m_ins)

    knn= NearestNeighbors(n_neighbors= neighbors)
    knn.fit(df_x)

    knn_index= knn.kneighbors(target_x, return_distance=False)
    
    trainX= df_x.drop(df_x.index[knn_index[0]])
    trainY= df_y.drop(df_y.index[knn_index[0]])

    testX= df_x.loc[df_x.index[knn_index[0]]]
    testY= df_y.loc[df_y.index[knn_index[0]]]

    testX= pd.concat([target_x, testX])
    testY= pd.concat([target_y, testY])
    
    return trainX, testX, trainY, testY

In [11]:
# find the neighb_size nearest neighbors from a target_x instance
# return data and target datsets based on knn
# with target_x and target_y into x_fts and y_trg

def knn_neighborhood(df_x, df_y, target_x, target_y, neighb_size= 0.2, target_in= True):
    
    m_ins= df_x.shape[0] # m x n dataset

    neighbors= math.floor(neighb_size* m_ins)

    knn= NearestNeighbors(n_neighbors= neighbors)
    knn.fit(df_x)

    knn_index= knn.kneighbors(target_x, return_distance=False)

    x_fts= df_x.loc[df_x.index[knn_index[0]]]
    y_trg= df_y.loc[df_y.index[knn_index[0]]]

    if (target_in== True):
        x_fts= pd.concat([target_x, x_fts])
        y_trg= pd.concat([target_y, y_trg])
    
    return x_fts, y_trg

In [13]:
# df_x and df_y doesn't contain target_x and target_y
# model is a a TRAINED model

# here we return p(x|i) and p(x|ij)
def knn_remove_and_return_costs(model, replace_ft_vals, x_tst, y_tst, target_x, target_y, 
                                neighb_size=0.01, verbose=False):
    
    start= time.time()

    m_ins= x_tst.shape[0] # m x n dataset
    n_fts= x_tst.shape[1]
    
    k_viz= math.floor(neighb_size* m_ins)
    k_viz_aux= math.floor(np.sqrt(m_ins))
    
    if (k_viz< k_viz_aux):
        k_viz= k_viz_aux
        
    # here, we generate a neighborhood of the target instance
    x_neighb, y_neighb= knn_neighborhood(x_tst, y_tst, target_x, target_y, neighb_size= (k_viz/ m_ins))    

    acc_no_i, acc_no_ij= remove_and_return_costs(model, replace_ft_vals, x_neighb, y_neighb)
    
    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))
    
    return acc_no_i, acc_no_ij

In [14]:
from sklearn.model_selection import KFold, StratifiedKFold

# df_x and df_y doesn't contain target_x and target_y
# model is a a TRAINED model

# here we return p(x|i) and p(x|ij)
def knnfold_remove_and_return_costs(model, replace_ft_vals, x_tst, y_tst, target_x, target_y, 
                                neighb_size=0.01, k_folds=5, verbose=False):
    
    start= time.time()

    m_ins= x_tst.shape[0] # m x n dataset
    n_fts= x_tst.shape[1]
    
    k_viz= math.floor(neighb_size* m_ins)
    k_viz_aux= math.floor(np.sqrt(m_ins))
    
    if (k_viz< k_viz_aux):
        k_viz= k_viz_aux
        
    # here, we generate a neighborhood of the target instance
    x_neighb, y_neighb= knn_neighborhood(x_tst, y_tst, target_x, target_y, 
                                         neighb_size= (k_viz/ m_ins), target_in= False) 
    

    K_folds= math.floor(np.sqrt(n_fts))

    if (K_folds< 5):
        K_folds= 5
        
    if (K_folds< k_folds):
        K_folds= k_folds   

    skf= StratifiedKFold(n_splits= K_folds, random_state=1234, shuffle=True)    
        
    acc_no_i= np.zeros(n_fts)
    acc_no_ij= np.zeros((n_fts,n_fts))
    
    for train_index, test_index in skf.split(x_neighb, y_neighb):
    
        x_fold= x_neighb.iloc[train_index]
        y_fold= y_neighb.iloc[train_index]
        
        # add the target instance on the fold set
        x_fold.append(target_x)
        y_fold.append(target_y)
    
        aux_acc_no_i, aux_acc_no_ij= remove_and_return_costs(model, replace_ft_vals, x_fold, y_fold)
    
        acc_no_i += aux_acc_no_i
        acc_no_ij += aux_acc_no_ij

    
    acc_no_i /= K_folds
    acc_no_ij /= K_folds
    
    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))
    
    return acc_no_i, acc_no_ij

In [15]:
# here we return | p(x|ij) - p(x|i) |

def get_p_matrix_v1(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pi)
                
    return p_matrix

In [16]:
# here we return | p(x|ij) - p(x|j) |

def get_p_matrix_v2(n_fts, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pj)
                
    return p_matrix

In [17]:
# here we return (| p(x|ij) - p(x|j) | + | p(x|i) - p(x) |) / 2

def get_p_matrix_v3(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pj)+ abs(pi- p))/ 2
                
    return p_matrix

In [18]:
# here we return (| p(x|ij) - p(x|i) | + | p(x|j) - p(x) |) / 2

def get_p_matrix_v4(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pi)+ abs(pj- p))/ 2
                
    return p_matrix

In [19]:
# the stationary distribution is the fraction of time that the system spends in each state as the 
# number of samples approaches infinity
# it looks like there's not a built-in method to find the stationary distribution

# converts a matrix to a row stochastic matrix - a real square matrix, with each row summing to 1
def to_row_stochastic_matrix(M):
    result= M
    
    for row in result:
        n= sum(row)
        if n> 0:
            row[:]= [f/sum(row) for f in row]
    
    return result

In [20]:
# the stationary distribution - analytical solution
# return 1D array

def stationary_dist_v1(stochastic_matrix):
    
    size_A= stochastic_matrix.shape[1]
    ones= [1]* size_A

    A= np.append(np.transpose(stochastic_matrix)- np.identity(size_A),[ones],axis=0)

    v= np.zeros(size_A+ 1)
    v[size_A]= 1
    v= np.transpose(v)

    stationary= np.linalg.solve(np.transpose(A).dot(A), np.transpose(A).dot(v))

    return stationary

In [21]:
# the stationary distribution - another analytical solution
# return 2D array

def stationary_dist_v2(stochastic_matrix):
    # we have to transpose so that Markov transitions correspond to right multiplying by a column vector
    eigval, eigvec= eigs(stochastic_matrix.T, k=1, which='LM')
    stationary= eigvec/ eigvec.sum()

    # eigs finds complex eigenvalues and eigenvectors, so you'll want the real part.
    stationary= stationary.real

    return stationary

In [22]:
# convert a p_matrix to a right stochastic matrix - a real square matrix, with each row summing to 1

def p_matrix_to_row_stochastic_matrix(p_matrix):
    
    st_matrix= np.asarray(p_matrix)
    
    # now convert to right stochastic matrix
    st_matrix= to_row_stochastic_matrix(st_matrix)
    
    return st_matrix

In [23]:
# find the stationary distribution
# return 1D array (version=1) or 2D array (version=1)

def p_matrix_to_stationary_dist(p_matrix, version=1):
    
    st_matrix= np.asarray(p_matrix)
    
    # now convert to right stochastic matrix
    st_matrix= to_row_stochastic_matrix(st_matrix)
    
    # then get the stationary distribution
    stationary_d= []
    
    if (version==1):
        stationary_d= stationary_dist_v1(st_matrix)
    else:
        stationary_d= stationary_dist_v2(st_matrix)
    
    return stationary_d

# New codes - Feature Attribution using Ranking

In [398]:
# here we return our simplified shapley value for one variable regarding first row of Shapley powerset

def get_p_fi(n_fts, fi, f0):
    
    n_term= (1/ n_fts) 
    
    phi_i= n_term* (fi- f0)
    
    return phi_i

In [399]:
def get_p_vector_fi(n_fts, acc_all, acc_no_i):
    p_vector= np.zeros((n_fts))
    f0= acc_all
    
    for i in range(n_fts):
        fi= acc_no_i[i]
        
        p_vector[i]= get_p_fi(n_fts, fi, f0)
        
    return p_vector

In [400]:
# here we return our simplified shapley value for two variables regarding second row of Shapley powerset

def get_p_fij(n_fts, fij, fi, fj, f0):
    
    n_term= n_fts* (n_fts- 1)
    n_term= 1/ n_term
    
    first_term= 2* fij
    scond_term= (n_fts- 2)* (fi+ fj)
    third_term= 2* (n_fts- 1)* f0
    
    phi_ij= n_term* (first_term+ scond_term- third_term)
    
    return phi_ij

In [401]:
def get_p_matrix_fij(n_fts, acc_all, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    f0= acc_all

    for i in range(n_fts):
        fi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                fj= acc_no_i[j]
                fij= acc_no_ij[i][j]
                
                p_matrix[i][j]= get_p_fij(n_fts, fij, fi, fj, f0)
                
    return p_matrix    

In [417]:
# here we return (p(x|ij) - p(x|i))^2

def get_p_matrix_div(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= np.power((pij- pi), 2)
                
    return p_matrix

In [416]:
# here we return shapley values phi_ij + phi_i

def get_sh_matrix_sum(n_fts, phi_i, phi_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= phi_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= phi_ij[i][j]

                p_matrix[i][j]= pi + pij
                
    return p_matrix

In [404]:
# Compute softmax values for each sets of scores in x
# x: numpy.ndarray, return numpy.ndarray
# x: pandas DataFrame, return pandas DataFrame

def softmax(x):
    
    npx= np.asarray(x)
    
    assert len(npx.shape)== 2
    s= np.max(npx, axis= 1)
    s= s[:, np.newaxis]
    e_x= np.exp(npx - s)
    div= np.sum(e_x, axis= 1)
    div= div[:, np.newaxis]
    
    result= e_x/ div
    
    dfx= pd.DataFrame(result)
    if (type(x)== type(dfx)):
        dfx.columns= x.columns
        return dfx
    
    return result

In [405]:
# normalize a numpy.ndarray to 0-1 range
# x: numpy.ndarray, return numpy.ndarray

def normalize(x):
    
    result= (x- np.min(x))/ (np.max(x)- np.min(x))
        
    return result

In [406]:
# return the mean prob from predicted probabilities, considering the class
# y_test: int numpy.ndarray
# y_pred: numpy.ndarray

def get_pred_proba(y_test, y_pred):
        
    assert len(y_test)== len(y_pred), "same size parameters expected"
    
    tam= len(y_pred)
    pred= 0
    
    for i in range(tam):
        pred= pred+ y_pred[i][y_test[i]]

    return (pred/ tam)

In [407]:
# remove WITH NO retraining - to speedup and stability improvements
# model is a TRAINED model

# NEW VERSION based on class predicted probabilities

# here we return p(x|i) and p(x|ij)
def remove_and_gen_costs_v2(model, n_fts, replace_ft_vals, x_tst, y_tst, use_preds= True, verbose=False):
    acc_no_i= []
    acc_no_ij= []

    start= time.time()

    for i in range(n_fts):  # remove and predict process to get gains/losses
        acc_row= []

        # replace the i-th ft of test set with its respective mode/mean to "remove" it.
        x_tst_copy_no_i= x_tst.copy()
        x_tst_copy_no_i.loc[:,x_tst_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        # if use_preds is True, predicted class labels are used. If false, the labels from y_tst are used
        if (use_preds== True):
            preds= model.predict(x_tst_copy_no_i)
            acc_no_i.append(get_pred_proba(preds, model.predict_proba(x_tst_copy_no_i)))
        else:
            acc_no_i.append(get_pred_proba(y_tst.values.ravel().astype(int),
                                           model.predict_proba(x_tst_copy_no_i)))
        
        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft of test with its respective mode/mean to "remove" it. 
                # here, we "remove" the j-th ft considering we have the i-th ft removed before
                # test the ML model and get the mean accuracy
                x_tst_copy_no_ij= x_tst_copy_no_i.copy()
                x_tst_copy_no_ij.loc[:,x_tst_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]
                
                # if use_preds is True, predicted class labels are used. If false, the labels from y_tst are used
                if (use_preds== True):
                    preds= model.predict(x_tst_copy_no_ij)
                    acc_row.append(get_pred_proba(preds, model.predict_proba(x_tst_copy_no_ij)))
                else:
                    acc_row.append(get_pred_proba(y_tst.values.ravel().astype(int), 
                                                   model.predict_proba(x_tst_copy_no_ij)))
               
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    
    if (verbose==True):
        print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

# testing

In [233]:
import synthetic_data as syn

synth= syn.synthetic_data_generator(n=500,df=True)

# split synth into features (x) and target (y)
x_syn= synth.loc[:,synth.columns[0:4]]
y_syn= synth.loc[:,synth.columns[4:5]]

x_syn.head()

,core_1,core_2,noise_1,noise_2
0,-2.827376,2.265393,2.011259,-2.890183
1,-1.450811,1.396575,1.183300,1.343149
2,-1.722920,2.319616,-0.098970,3.198920
3,-2.312536,2.530822,-3.412652,-3.186505
4,-1.851225,3.461515,0.221592,2.182029


In [234]:
soft_x_syn= softmax(x_syn)

soft_x_syn.head()

,core_1,core_2,noise_1,noise_2
0,0.003436,0.559442,0.433896,0.003226
1,0.020610,0.355378,0.287122,0.336890
2,0.004993,0.284429,0.025328,0.685251
3,0.007773,0.986396,0.002587,0.003244
4,0.003727,0.756272,0.029621,0.210380


In [375]:
norm_x_syn= normalize_selected(x_syn)

norm_x_syn.head()

,core_1,core_2,noise_1,noise_2
0,0.321708,0.687528,0.751470,0.138741
1,0.428056,0.612178,0.647921,0.668081
2,0.407034,0.692231,0.487554,0.900128
3,0.361482,0.710548,0.073129,0.101689
4,0.397121,0.791264,0.527645,0.772975


In [294]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(soft_x_syn,y_syn,
                                                                                 train_size=0.80,
                                                                                 random_state=1234)

xgb_model= xgb.XGBRFClassifier(learning_rate= 0.01,
                               n_estimators= 500,
                               max_depth= 6,
                               gamma= 1,
                               objective= 'binary:logistic',
                               eval_metric='logloss',
                               use_label_encoder=False)

eval_set= [(train, labels_train), (test, labels_test)]
eval_metric= ["auc","error"]

xgb_model.fit(train, labels_train, eval_metric=eval_metric, eval_set=eval_set, verbose=False)

acc_all= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))
acc_all

0.955

In [295]:
proba_all= get_pred_proba(labels_test.values.ravel().astype(int),xgb_model.predict_proba(test))
proba_all

0.5043042317032814

In [296]:
proba_all_preds= get_pred_proba(xgb_model.predict(test), xgb_model.predict_proba(test))
proba_all_preds

0.5044857385754585

In [297]:
num_fts= len(x_syn.columns)

replace_ft= replace_values(soft_x_syn, soft_x_syn.columns, num_type='mean')
replace_ft

,core_1,core_2,noise_1,noise_2
0,0.110713,0.375399,0.256847,0.257041


In [268]:
xgb_model.predict(replace_ft)

array([1])

In [269]:
xgb_model.predict_proba(replace_ft)

array([[0.49501997, 0.50498   ]], dtype=float32)

In [305]:
# the instance to be explained (from train dataset)
target_pos= 1

target_inst= train.iloc[target_pos].to_frame().T
target_labl= labels_train.iloc[target_pos].to_frame().T

target_inst

,core_1,core_2,noise_1,noise_2
42,0.000859,0.166143,0.796697,0.036301


In [306]:
num_fts= len(x_syn.columns)

acc_no_i, acc_no_ij= remove_and_gen_costs_v2(xgb_model, num_fts, replace_ft, target_inst, target_labl, 
                                            use_preds= False, verbose=True)

--- 0.17 seconds ---


In [418]:
# get the probability matrix
p_matrix= get_p_matrix_div(num_fts, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary= p_matrix_to_stationary_dist(p_matrix)

ft_ranking= pd.Series(stationary, index=train.columns).sort_values(ascending=False)
ft_ranking

core_1     0.499999
core_2     0.493731
noise_1    0.006175
noise_2    0.000095
dtype: float64

In [419]:
p_matrix

array([[0.00000000e+00, 9.87465255e-01, 1.23482906e-02, 1.86454072e-04],
       [9.99994798e-01, 0.00000000e+00, 2.41483220e-06, 2.78713353e-06],
       [9.99999638e-01, 3.62283173e-07, 0.00000000e+00, 0.00000000e+00],
       [9.99998113e-01, 0.00000000e+00, 1.88655108e-06, 0.00000000e+00]])

In [414]:
# get the first row shapley values to fi
sh_phi_i= get_p_vector_fi(num_fts, proba_all, acc_no_i)

# get the second row shapley values to fij
sh_phi_ij= get_p_matrix_fij(num_fts, proba_all, acc_no_i, acc_no_ij)

# get the probability matrix divergence between fij and fi
p_matrix= normalize(get_sh_matrix_sum(num_fts, sh_phi_i, sh_phi_ij))

# get the stationary distribution
stationary= p_matrix_to_stationary_dist(p_matrix)

ft_ranking= pd.Series(stationary, index=train.columns).sort_values(ascending=False)
ft_ranking

core_1     0.530602
core_2     0.194284
noise_2    0.140057
noise_1    0.135057
dtype: float64

In [415]:
get_sh_matrix_sum(num_fts, sh_phi_i, sh_phi_ij)

array([[ 0.        , -0.00423893, -0.00487622, -0.00482205],
       [-0.00195927,  0.        ,  0.00050861,  0.00050346],
       [-0.00259578,  0.00050939,  0.        ,  0.00050685],
       [-0.00254476,  0.0005011 ,  0.00050371,  0.        ]])

# Feature Attribution using PageRank - Tests

In [26]:
def run_lib_pr(graph_matrix, ft_names, iteration= 100, damping_factor= 0.85, tolerance= 1.0e-6):
    
    num_fts= graph_matrix.shape[0]
    
    D= nx.DiGraph()

    for i in range(num_fts):
        for j in range(num_fts):
            if (i!= j):
                D.add_weighted_edges_from([(ft_names[i],ft_names[j],graph_matrix[i,j])])
    
                
    pRank= pd.Series(nx.pagerank(D, max_iter=iteration, alpha=damping_factor, tol=tolerance))

    return pRank.sort_values(ascending=False)

In [27]:
# using my PR to compare results
def run_my_pr(graph_matrix, ft_names, iteration= 100, damping_factor= 0.95, tolerance= 1.0e-6):
    
    file_path= 'datasets/FAR_data.txt'
    
    num_fts= graph_matrix.shape[0]

    f= open(file_path, 'w')

    for i in range(num_fts):
        for j in range(num_fts):
            line= (str(int(i)) + ',' + str(int(j)) + ',' + str(graph_matrix[i,j]) + '\n')
            f.write(line)

    f.close()

    graph= gpr.init_graph(file_path)

    myPRank= pd.Series(pr.run_PageRank(graph, iteration=iteration, damping_factor=damping_factor, 
                                       tolerance=tolerance))
    myPRank= myPRank.sort_values(ascending=False)
    
    ids_names= ft_names[(myPRank.index).astype(int)]

    myPRank.index= ids_names

    return myPRank

# --- Tests using the Titanic dataset ---

# Data loading and preprocessing

In [28]:
#!kaggle competitions download -c titanic

In [28]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [29]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [30]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [31]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S
1306,1,female,39.0,0,0,108.9000,C
1307,3,male,38.5,0,0,7.2500,S


In [32]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [34]:
# in this case we'll only use X_train df because X_test is not labeled

In [33]:
X_train= pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')

X_train= pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S
888,1,female,19.0,0,0,30.0000,S
889,3,female,28.0,1,2,23.4500,S


In [34]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [35]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [36]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize_selected(X_train, numeric_columns)
X_train_ohe= normalize_selected(X_train_ohe, numeric_columns)

# ML model setup

In [37]:
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

# --- Run global modeling (no retraining) ---

In [40]:
repeat_train= 1

num_fts= len(train.columns)

In [43]:
# train the ML model and get the accuracy using the entire feature set
acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

# rf is the model trained

acc_all_fts

0.8212290502793296

In [44]:
# values to "remove" (entire dataset)
replace_ft= replace_values(X_train_ohe, numeric_columns, num_type='mean', cat_type='median')

replace_ft

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.363679,0.065376,0.063599,0.062858,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [46]:
# calculates "transition" costs removing columns from the feature set using a trained model

acc_no_i, acc_no_ij= remove_and_return_costs(rf, replace_ft, test, labels_test, True)

--- 13.36 seconds ---


In [47]:
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

[0.777 0.816 0.81  0.81  0.827 0.827 0.821 0.771 0.771 0.832 0.821 0.838]
-----------------------------
[[0.    0.81  0.777 0.804 0.782 0.777 0.749 0.726 0.709 0.777 0.788 0.788]
 [0.81  0.    0.816 0.81  0.832 0.816 0.793 0.754 0.76  0.821 0.816 0.832]
 [0.777 0.816 0.    0.81  0.816 0.816 0.81  0.749 0.743 0.838 0.81  0.838]
 [0.804 0.81  0.81  0.    0.821 0.81  0.81  0.771 0.788 0.81  0.81  0.799]
 [0.782 0.832 0.816 0.821 0.    0.832 0.749 0.777 0.777 0.821 0.827 0.827]
 [0.777 0.816 0.816 0.81  0.832 0.    0.749 0.76  0.754 0.838 0.827 0.844]
 [0.749 0.793 0.81  0.81  0.749 0.749 0.    0.726 0.726 0.816 0.816 0.81 ]
 [0.726 0.754 0.749 0.771 0.777 0.76  0.726 0.    0.648 0.777 0.765 0.771]
 [0.709 0.76  0.743 0.788 0.777 0.754 0.726 0.648 0.    0.782 0.765 0.782]
 [0.777 0.821 0.838 0.81  0.821 0.838 0.816 0.777 0.782 0.    0.832 0.827]
 [0.788 0.816 0.81  0.81  0.827 0.827 0.816 0.765 0.765 0.832 0.    0.832]
 [0.788 0.832 0.838 0.799 0.827 0.844 0.81  0.771 0.782 0.827 0.832 0. 

In [48]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.034 0.    0.028 0.006 0.    0.028 0.05  0.067 0.    0.011 0.011]
 [0.006 0.    0.    0.006 0.017 0.    0.022 0.061 0.056 0.006 0.    0.017]
 [0.034 0.006 0.    0.    0.006 0.006 0.    0.061 0.067 0.028 0.    0.028]
 [0.006 0.    0.    0.    0.011 0.    0.    0.039 0.022 0.    0.    0.011]
 [0.045 0.006 0.011 0.006 0.    0.006 0.078 0.05  0.05  0.006 0.    0.   ]
 [0.05  0.011 0.011 0.017 0.006 0.    0.078 0.067 0.073 0.011 0.    0.017]
 [0.073 0.028 0.011 0.011 0.073 0.073 0.    0.095 0.095 0.006 0.006 0.011]
 [0.045 0.017 0.022 0.    0.006 0.011 0.045 0.    0.123 0.006 0.006 0.   ]
 [0.061 0.011 0.028 0.017 0.006 0.017 0.045 0.123 0.    0.011 0.006 0.011]
 [0.056 0.011 0.006 0.022 0.011 0.006 0.017 0.056 0.05  0.    0.    0.006]
 [0.034 0.006 0.011 0.011 0.006 0.006 0.006 0.056 0.056 0.011 0.    0.011]
 [0.05  0.006 0.    0.039 0.011 0.006 0.028 0.067 0.056 0.011 0.006 0.   ]]
-----------------------------
[[0.    0.006 0.034 0.006 0.045 0.05  0.073 0.045 0.061 0.056 0.034 0

In [49]:
# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.13374431 0.05231228 0.0440794  0.04280727 0.04367295 0.0415467
 0.11627019 0.21858259 0.23010968 0.02478541 0.01663662 0.0354526 ]
-----------------------------
[0.07893536 0.05345958 0.06962461 0.02270331 0.08196865 0.10497604
 0.1789445  0.09831183 0.11683528 0.05951818 0.05837211 0.07635055]
-----------------------------
[0.09003684 0.06262179 0.06440699 0.04254829 0.08535842 0.09727546
 0.14812166 0.09346252 0.103499   0.06624631 0.06402856 0.08239416]
-----------------------------
[0.16281147 0.04517182 0.05079278 0.05195564 0.03447498 0.03509667
 0.07181082 0.21690014 0.22574803 0.03896661 0.01103358 0.05523747]


In [50]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [51]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.230110
Sex_female    0.218583
Age           0.133744
Pclass_3      0.116270
SibSp         0.052312
Parch         0.044079
Pclass_1      0.043673
Fare          0.042807
Pclass_2      0.041547
Embarked_S    0.035453
Embarked_C    0.024785
Embarked_Q    0.016637
dtype: float64

In [52]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.448692
1,Pclass,0.201490
2,Age,0.133744
3,Embarked,0.076875
4,SibSp,0.052312
5,Parch,0.044079
6,Fare,0.042807


In [53]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Pclass_3      0.178945
Sex_male      0.116835
Pclass_2      0.104976
Sex_female    0.098312
Pclass_1      0.081969
Age           0.078935
Embarked_S    0.076351
Parch         0.069625
Embarked_C    0.059518
Embarked_Q    0.058372
SibSp         0.053460
Fare          0.022703
dtype: float64

In [54]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Pclass,0.365889
1,Sex,0.215147
2,Embarked,0.194241
3,Age,0.078935
4,Parch,0.069625
5,SibSp,0.053460
6,Fare,0.022703


In [55]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Pclass_3      0.148122
Sex_male      0.103499
Pclass_2      0.097275
Sex_female    0.093463
Age           0.090037
Pclass_1      0.085358
Embarked_S    0.082394
Embarked_C    0.066246
Parch         0.064407
Embarked_Q    0.064029
SibSp         0.062622
Fare          0.042548
dtype: float64

In [56]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.448692
1,Pclass,0.201490
2,Age,0.133744
3,Embarked,0.076875
4,SibSp,0.052312
5,Parch,0.044079
6,Fare,0.042807


In [57]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.225748
Sex_female    0.216900
Age           0.162811
Pclass_3      0.071811
Embarked_S    0.055237
Fare          0.051956
Parch         0.050793
SibSp         0.045172
Embarked_C    0.038967
Pclass_2      0.035097
Pclass_1      0.034475
Embarked_Q    0.011034
dtype: float64

In [58]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.448692
1,Pclass,0.201490
2,Age,0.133744
3,Embarked,0.076875
4,SibSp,0.052312
5,Parch,0.044079
6,Fare,0.042807


# --- Run local modeling (k-NN version, no retraining) ---

In [64]:
# the instance to be explained (from train dataset)
target_index= 1

target_inst= train.loc[train.index== target_index]
target_labl= labels_train.loc[labels_train.index== target_index]

In [72]:
# calculates "transition" costs removing columns from a target instance using a trained model
# target instance is supported by neighb_size percent nearest neighbors of test dataset

acc_no_i, acc_no_ij= knn_remove_and_return_costs(rf, replace_ft, test, labels_test, target_inst, target_labl,
                                                 neighb_size= 0.5, verbose=True)

--- 12.29 seconds ---


In [73]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8212290502793296
-----------------------------
[0.778 0.844 0.833 0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.833 0.856]
-----------------------------
[[0.    0.844 0.778 0.833 0.767 0.778 0.778 0.767 0.767 0.789 0.822 0.833]
 [0.844 0.    0.856 0.822 0.844 0.844 0.844 0.822 0.822 0.844 0.844 0.844]
 [0.778 0.856 0.    0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.833 0.856]
 [0.833 0.822 0.833 0.    0.833 0.833 0.833 0.833 0.833 0.833 0.833 0.833]
 [0.767 0.844 0.822 0.833 0.    0.833 0.844 0.8   0.8   0.844 0.822 0.844]
 [0.778 0.844 0.844 0.833 0.833 0.    0.833 0.822 0.822 0.867 0.844 0.867]
 [0.778 0.844 0.833 0.833 0.844 0.833 0.    0.811 0.811 0.856 0.833 0.856]
 [0.767 0.822 0.811 0.833 0.8   0.822 0.811 0.    0.8   0.833 0.811 0.833]
 [0.767 0.822 0.811 0.833 0.8   0.822 0.811 0.8   0.    0.833 0.811 0.833]
 [0.789 0.844 0.856 0.833 0.844 0.867 0.856 0.833 0.833 0.    0.856 0.856]
 [0.822 0.844 0.833 0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.    0.856]
 [0.833 0.844 0.856 0.

In [74]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.067 0.    0.056 0.011 0.    0.    0.011 0.011 0.011 0.044 0.056]
 [0.    0.    0.011 0.022 0.    0.    0.    0.022 0.022 0.    0.    0.   ]
 [0.056 0.022 0.    0.    0.011 0.011 0.    0.022 0.022 0.022 0.    0.022]
 [0.    0.011 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.056 0.022 0.    0.011 0.    0.011 0.022 0.022 0.022 0.022 0.    0.022]
 [0.067 0.    0.    0.011 0.011 0.    0.011 0.022 0.022 0.022 0.    0.022]
 [0.056 0.011 0.    0.    0.011 0.    0.    0.022 0.022 0.022 0.    0.022]
 [0.044 0.011 0.    0.022 0.011 0.011 0.    0.    0.011 0.022 0.    0.022]
 [0.044 0.011 0.    0.022 0.011 0.011 0.    0.011 0.    0.022 0.    0.022]
 [0.067 0.011 0.    0.022 0.011 0.011 0.    0.022 0.022 0.    0.    0.   ]
 [0.011 0.011 0.    0.    0.011 0.011 0.    0.022 0.022 0.022 0.    0.022]
 [0.022 0.011 0.    0.022 0.011 0.011 0.    0.022 0.022 0.    0.    0.   ]]
-----------------------------
[[0.    0.    0.056 0.    0.056 0.067 0.056 0.044 0.044 0.067 0.011 0

In [75]:
# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.13640016 0.22082736 0.03154677 0.14958462 0.03881635 0.03284285
 0.00601786 0.11457667 0.11457667 0.05467199 0.02273336 0.07740535]
-----------------------------
[0.15272172 0.10897663 0.0924549  0.00641039 0.12706025 0.09467783
 0.07251911 0.07095402 0.07095402 0.07468928 0.06996854 0.05861331]
-----------------------------
[0.11154369 0.07064625 0.08635688 0.05211477 0.09598311 0.08954544
 0.0828273  0.082535   0.082535   0.08495693 0.08082021 0.0801354 ]
-----------------------------
[0.18407963 0.10824051 0.03810719 0.08900132 0.02567714 0.07819913
 0.03732032 0.07189512 0.07189512 0.11464074 0.0513386  0.12960519]


In [76]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [77]:
# TARGET INSTANCE TO EXPLAIN ITS FEATURES

X_all.loc[X_all.index== target_index]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.25,S


In [78]:
target_label

PassengerId
1    0
Name: Survived, dtype: int64

In [79]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

SibSp         0.220827
Fare          0.149585
Age           0.136400
Sex_male      0.114577
Sex_female    0.114577
Embarked_S    0.077405
Embarked_C    0.054672
Pclass_1      0.038816
Pclass_2      0.032843
Parch         0.031547
Embarked_Q    0.022733
Pclass_3      0.006018
dtype: float64

In [80]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.229153
1,SibSp,0.220827
2,Embarked,0.154811
3,Fare,0.149585
4,Age,0.136400
5,Pclass,0.077677
6,Parch,0.031547


In [81]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.152722
Pclass_1      0.127060
SibSp         0.108977
Pclass_2      0.094678
Parch         0.092455
Embarked_C    0.074689
Pclass_3      0.072519
Sex_male      0.070954
Sex_female    0.070954
Embarked_Q    0.069969
Embarked_S    0.058613
Fare          0.006410
dtype: float64

In [82]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Pclass,0.294257
1,Embarked,0.203271
2,Age,0.152722
3,Sex,0.141908
4,SibSp,0.108977
5,Parch,0.092455
6,Fare,0.006410


In [83]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.111544
Pclass_1      0.095983
Pclass_2      0.089545
Parch         0.086357
Embarked_C    0.084957
Pclass_3      0.082827
Sex_female    0.082535
Sex_male      0.082535
Embarked_Q    0.080820
Embarked_S    0.080135
SibSp         0.070646
Fare          0.052115
dtype: float64

In [84]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.229153
1,SibSp,0.220827
2,Embarked,0.154811
3,Fare,0.149585
4,Age,0.136400
5,Pclass,0.077677
6,Parch,0.031547


In [85]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.184080
Embarked_S    0.129605
Embarked_C    0.114641
SibSp         0.108241
Fare          0.089001
Pclass_2      0.078199
Sex_female    0.071895
Sex_male      0.071895
Embarked_Q    0.051339
Parch         0.038107
Pclass_3      0.037320
Pclass_1      0.025677
dtype: float64

In [86]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.229153
1,SibSp,0.220827
2,Embarked,0.154811
3,Fare,0.149585
4,Age,0.136400
5,Pclass,0.077677
6,Parch,0.031547


# --- Run PageRank approach (k-NN local version, no retraining) ---

In [87]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8212290502793296
-----------------------------
[0.778 0.844 0.833 0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.833 0.856]
-----------------------------
[[0.    0.844 0.778 0.833 0.767 0.778 0.778 0.767 0.767 0.789 0.822 0.833]
 [0.844 0.    0.856 0.822 0.844 0.844 0.844 0.822 0.822 0.844 0.844 0.844]
 [0.778 0.856 0.    0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.833 0.856]
 [0.833 0.822 0.833 0.    0.833 0.833 0.833 0.833 0.833 0.833 0.833 0.833]
 [0.767 0.844 0.822 0.833 0.    0.833 0.844 0.8   0.8   0.844 0.822 0.844]
 [0.778 0.844 0.844 0.833 0.833 0.    0.833 0.822 0.822 0.867 0.844 0.867]
 [0.778 0.844 0.833 0.833 0.844 0.833 0.    0.811 0.811 0.856 0.833 0.856]
 [0.767 0.822 0.811 0.833 0.8   0.822 0.811 0.    0.8   0.833 0.811 0.833]
 [0.767 0.822 0.811 0.833 0.8   0.822 0.811 0.8   0.    0.833 0.811 0.833]
 [0.789 0.844 0.856 0.833 0.844 0.867 0.856 0.833 0.833 0.    0.856 0.856]
 [0.822 0.844 0.833 0.833 0.822 0.844 0.833 0.811 0.811 0.856 0.    0.856]
 [0.833 0.844 0.856 0.

In [88]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= p_matrix_to_row_stochastic_matrix(p_matrix1)
st_matrix2= p_matrix_to_row_stochastic_matrix(p_matrix2)
st_matrix3= p_matrix_to_row_stochastic_matrix(p_matrix3)
st_matrix4= p_matrix_to_row_stochastic_matrix(p_matrix4)

In [89]:
run_lib_pr(st_matrix1, X_train_ohe.columns)

SibSp         0.189767
Age           0.138879
Fare          0.133546
Sex_female    0.108568
Sex_male      0.108568
Embarked_S    0.082153
Embarked_C    0.062479
Pclass_1      0.047703
Pclass_2      0.041762
Parch         0.035543
Embarked_Q    0.032175
Pclass_3      0.018856
dtype: float64

In [90]:
run_lib_pr(st_matrix2, X_train_ohe.columns)

Age           0.145641
Pclass_1      0.120861
SibSp         0.104390
Pclass_2      0.093024
Parch         0.089839
Embarked_C    0.075503
Pclass_3      0.073324
Sex_male      0.072818
Sex_female    0.072818
Embarked_Q    0.071589
Embarked_S    0.062472
Fare          0.017719
dtype: float64

In [91]:
run_lib_pr(st_matrix3, X_train_ohe.columns)

Age           0.108278
Pclass_1      0.094095
Pclass_2      0.088436
Parch         0.085675
Embarked_C    0.084639
Pclass_3      0.082627
Sex_female    0.082626
Sex_male      0.082626
Embarked_Q    0.081004
Embarked_S    0.080636
SibSp         0.072678
Fare          0.056679
dtype: float64

In [92]:
run_lib_pr(st_matrix4, X_train_ohe.columns)

Age           0.171707
Embarked_S    0.123069
Embarked_C    0.111070
SibSp         0.103678
Fare          0.086359
Pclass_2      0.078944
Sex_male      0.073619
Sex_female    0.073619
Embarked_Q    0.054886
Parch         0.044465
Pclass_3      0.044310
Pclass_1      0.034274
dtype: float64

In [93]:
run_my_pr(st_matrix1, X_train_ohe.columns)

Age           0.182341
SibSp         0.168903
Fare          0.094953
Sex_female    0.080114
Sex_male      0.075810
Pclass_1      0.074361
Pclass_2      0.067803
Embarked_C    0.057917
Embarked_S    0.053844
Embarked_Q    0.048640
Parch         0.048144
Pclass_3      0.047169
dtype: float64

In [94]:
run_my_pr(st_matrix2, X_train_ohe.columns)

Age           0.172892
SibSp         0.137493
Pclass_1      0.119340
Parch         0.091126
Pclass_2      0.089096
Pclass_3      0.069556
Sex_female    0.062003
Sex_male      0.059481
Embarked_Q    0.053577
Embarked_C    0.050862
Embarked_S    0.049421
Fare          0.045154
dtype: float64

In [95]:
run_my_pr(st_matrix3, X_train_ohe.columns)

Age           0.145351
SibSp         0.103802
Parch         0.101680
Pclass_1      0.099846
Pclass_2      0.087622
Pclass_3      0.078158
Fare          0.074444
Sex_female    0.070383
Sex_male      0.066671
Embarked_C    0.060240
Embarked_Q    0.057898
Embarked_S    0.053905
dtype: float64

In [96]:
run_my_pr(st_matrix4, X_train_ohe.columns)

Age           0.208401
SibSp         0.112831
Fare          0.085504
Pclass_2      0.082193
Sex_female    0.070637
Embarked_C    0.069816
Parch         0.068458
Sex_male      0.067481
Pclass_1      0.062262
Pclass_3      0.059714
Embarked_S    0.058974
Embarked_Q    0.053730
dtype: float64

# --- Run local modeling (k-Fold version, no retraining) ---

In [99]:
# calculates "transition" costs removing columns from a target instance using a trained model
# target instance is supported by neighb_size percent nearest neighbors of test dataset applied in a kfolding way

acc_no_i, acc_no_ij= knnfold_remove_and_return_costs(rf, replace_ft, test, labels_test, 
                                                     target_inst, target_labl, neighb_size= 0.5, 
                                                     k_folds=10, verbose=True)

--- 126.52 seconds ---


In [100]:
print(acc_all_fts)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8212290502793296
-----------------------------
[0.775 0.843 0.831 0.831 0.82  0.843 0.831 0.809 0.809 0.854 0.831 0.854]
-----------------------------
[[0.    0.843 0.775 0.831 0.764 0.775 0.775 0.764 0.764 0.787 0.82  0.831]
 [0.843 0.    0.854 0.82  0.843 0.843 0.843 0.82  0.82  0.843 0.843 0.843]
 [0.775 0.854 0.    0.831 0.82  0.843 0.831 0.809 0.809 0.854 0.831 0.854]
 [0.831 0.82  0.831 0.    0.831 0.831 0.831 0.831 0.831 0.831 0.831 0.831]
 [0.764 0.843 0.82  0.831 0.    0.831 0.843 0.798 0.798 0.843 0.82  0.843]
 [0.775 0.843 0.843 0.831 0.831 0.    0.831 0.82  0.82  0.865 0.843 0.865]
 [0.775 0.843 0.831 0.831 0.843 0.831 0.    0.809 0.809 0.854 0.831 0.854]
 [0.764 0.82  0.809 0.831 0.798 0.82  0.809 0.    0.798 0.831 0.809 0.831]
 [0.764 0.82  0.809 0.831 0.798 0.82  0.809 0.798 0.    0.831 0.809 0.831]
 [0.787 0.843 0.854 0.831 0.843 0.865 0.854 0.831 0.831 0.    0.854 0.854]
 [0.82  0.843 0.831 0.831 0.82  0.843 0.831 0.809 0.809 0.854 0.    0.854]
 [0.831 0.843 0.854 0.

In [101]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.067 0.    0.056 0.011 0.    0.    0.011 0.011 0.011 0.045 0.056]
 [0.    0.    0.011 0.022 0.    0.    0.    0.022 0.022 0.    0.    0.   ]
 [0.056 0.022 0.    0.    0.011 0.011 0.    0.022 0.022 0.022 0.    0.022]
 [0.    0.011 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.056 0.022 0.    0.011 0.    0.011 0.022 0.022 0.022 0.022 0.    0.022]
 [0.067 0.    0.    0.011 0.011 0.    0.011 0.022 0.022 0.022 0.    0.022]
 [0.056 0.011 0.    0.    0.011 0.    0.    0.022 0.022 0.022 0.    0.022]
 [0.045 0.011 0.    0.022 0.011 0.011 0.    0.    0.011 0.022 0.    0.022]
 [0.045 0.011 0.    0.022 0.011 0.011 0.    0.011 0.    0.022 0.    0.022]
 [0.067 0.011 0.    0.022 0.011 0.011 0.    0.022 0.022 0.    0.    0.   ]
 [0.011 0.011 0.    0.    0.011 0.011 0.    0.022 0.022 0.022 0.    0.022]
 [0.022 0.011 0.    0.022 0.011 0.011 0.    0.022 0.022 0.    0.    0.   ]]
-----------------------------
[[0.    0.    0.056 0.    0.056 0.067 0.056 0.045 0.045 0.067 0.011 0

In [102]:
# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.1367098  0.22044151 0.03169028 0.14951597 0.03883469 0.03284883
 0.00601969 0.11449862 0.11449862 0.05468602 0.02278497 0.07747099]
-----------------------------
[0.15271816 0.1088576  0.09244082 0.00653571 0.12704896 0.09467993
 0.0725112  0.07096152 0.07096152 0.07469304 0.06996769 0.05862384]
-----------------------------
[0.11447102 0.07072906 0.08581853 0.0518159  0.09684943 0.08948847
 0.08211171 0.0822791  0.0822791  0.08433869 0.08031249 0.0795065 ]
-----------------------------
[0.19203229 0.10702169 0.03352095 0.08773161 0.02629028 0.07501024
 0.03268352 0.07810286 0.07810286 0.11262367 0.04803437 0.12884567]


In [103]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [104]:
# TARGET INSTANCE TO EXPLAIN ITS FEATURES

X_all.loc[X_all.index== target_index]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.25,S


In [105]:
target_label

PassengerId
1    0
Name: Survived, dtype: int64

In [106]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

SibSp         0.220442
Fare          0.149516
Age           0.136710
Sex_female    0.114499
Sex_male      0.114499
Embarked_S    0.077471
Embarked_C    0.054686
Pclass_1      0.038835
Pclass_2      0.032849
Parch         0.031690
Embarked_Q    0.022785
Pclass_3      0.006020
dtype: float64

In [107]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.228997
1,SibSp,0.220442
2,Embarked,0.154942
3,Fare,0.149516
4,Age,0.136710
5,Pclass,0.077703
6,Parch,0.031690


In [108]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.152718
Pclass_1      0.127049
SibSp         0.108858
Pclass_2      0.094680
Parch         0.092441
Embarked_C    0.074693
Pclass_3      0.072511
Sex_female    0.070962
Sex_male      0.070962
Embarked_Q    0.069968
Embarked_S    0.058624
Fare          0.006536
dtype: float64

In [109]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Pclass,0.294240
1,Embarked,0.203285
2,Age,0.152718
3,Sex,0.141923
4,SibSp,0.108858
5,Parch,0.092441
6,Fare,0.006536


In [110]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.114471
Pclass_1      0.096849
Pclass_2      0.089488
Parch         0.085819
Embarked_C    0.084339
Sex_female    0.082279
Sex_male      0.082279
Pclass_3      0.082112
Embarked_Q    0.080312
Embarked_S    0.079506
SibSp         0.070729
Fare          0.051816
dtype: float64

In [111]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.228997
1,SibSp,0.220442
2,Embarked,0.154942
3,Fare,0.149516
4,Age,0.136710
5,Pclass,0.077703
6,Parch,0.031690


In [112]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.192032
Embarked_S    0.128846
Embarked_C    0.112624
SibSp         0.107022
Fare          0.087732
Sex_female    0.078103
Sex_male      0.078103
Pclass_2      0.075010
Embarked_Q    0.048034
Parch         0.033521
Pclass_3      0.032684
Pclass_1      0.026290
dtype: float64

In [113]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.228997
1,SibSp,0.220442
2,Embarked,0.154942
3,Fare,0.149516
4,Age,0.136710
5,Pclass,0.077703
6,Parch,0.031690


# --- Global version with retraining ---

In [115]:
repeat_train= 5

acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

acc_no_i, acc_no_ij= remove_and_retrain(rf, replace_ft, train, test, labels_train, labels_test,
                                        repeat_train, verbose=True)

--- 839.55 seconds ---


In [117]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= p_matrix_to_stationary_dist(p_matrix4)

In [118]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [119]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.180684
Sex_male      0.122178
Sex_female    0.116498
Parch         0.107287
Fare          0.102084
SibSp         0.079221
Embarked_S    0.055937
Pclass_3      0.050458
Pclass_2      0.048057
Embarked_Q    0.047636
Pclass_1      0.047170
Embarked_C    0.042790
dtype: float64

In [120]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.238676
1,Age,0.180684
2,Embarked,0.146362
3,Pclass,0.145685
4,Parch,0.107287
5,Fare,0.102084
6,SibSp,0.079221


In [121]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.165706
Sex_male      0.164506
Parch         0.097445
Pclass_3      0.088626
Age           0.086348
Pclass_2      0.078458
Embarked_S    0.067442
Embarked_C    0.056864
Fare          0.053951
Pclass_1      0.052439
SibSp         0.048966
Embarked_Q    0.039250
dtype: float64

In [122]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.330211
1,Pclass,0.219523
2,Embarked,0.163556
3,Parch,0.097445
4,Age,0.086348
5,Fare,0.053951
6,SibSp,0.048966


In [123]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.198125
Sex_female    0.192781
Pclass_3      0.075483
Age           0.071329
Parch         0.071310
Fare          0.062194
SibSp         0.061153
Pclass_2      0.058004
Embarked_S    0.057029
Pclass_1      0.055414
Embarked_C    0.054577
Embarked_Q    0.042602
dtype: float64

In [124]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.238676
1,Age,0.180684
2,Embarked,0.146362
3,Pclass,0.145685
4,Parch,0.107287
5,Fare,0.102084
6,SibSp,0.079221


In [125]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.228860
Fare          0.125285
Parch         0.120473
Pclass_1      0.080464
Embarked_S    0.071641
Sex_male      0.068547
SibSp         0.058951
Pclass_3      0.058375
Sex_female    0.058265
Embarked_Q    0.047798
Pclass_2      0.044935
Embarked_C    0.036407
dtype: float64

In [126]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.238676
1,Age,0.180684
2,Embarked,0.146362
3,Pclass,0.145685
4,Parch,0.107287
5,Fare,0.102084
6,SibSp,0.079221


# Misc

In [88]:
eigval, eigvec= eigs(st_matrix3.T, k=1, which='LM')
eigval

array([1.+0.j])

In [89]:
f= open('datasets/FAR_data.txt', 'w')

for i in range(num_fts):
    for j in range(num_fts):
        line= (str(i) + ',' + str(j) + ',' + str(p_matrix1[i][j]) + '\n')
        f.write(line)
        
f.close()